# Small Language Model Evaluation: Controlled Instruction Following Study

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`

A reproducible exploratory comparison of baseline prompting and a concise system instruction across ten fixed prompts. Recorded results and a separately completed provisional human audit are provided in the repository. This notebook generates the experiment outputs; the final report incorporates the subsequent human audit.


## 1. Set up the environment

Run the notebook from top to bottom in a fresh Google Colab runtime. A GPU runtime is recommended (`Runtime → Change runtime type → T4 GPU` if available). The model is small, but downloading the model requires an internet connection and the first run may take several minutes.


In [ ]:
%pip -q install -U transformers accelerate pandas matplotlib


In [ ]:
import os, json, time, random, platform, re, zipfile, sys
from pathlib import Path
from difflib import SequenceMatcher
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 20261018
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 160
TEMPERATURE = 0.2
TOP_P = 0.9
DO_SAMPLE = True
OUTPUT_DIR = Path("slm_evaluation_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

runtime_info = {
    "model_id": MODEL_ID,
    "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "do_sample": DO_SAMPLE,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
}
print(json.dumps(runtime_info, indent=2))
with open(OUTPUT_DIR / "runtime_metadata.json", "w", encoding="utf-8") as f:
    json.dump(runtime_info, f, indent=2)


## 2. Define the test set

The prompts cover explanation, classification, arithmetic, data leakage, strict formatting, basic coding and uncertainty. Expected-answer notes are evaluation guidance, not text shown to the model.


In [ ]:
TEST_CASES = [
    {"id": 1, "category": "explanation", "prompt": "Explain overfitting in machine learning using a simple example.", "expected": "Explains fitting training data too closely and poorer performance on new data; gives a clear example."},
    {"id": 2, "category": "concept_comparison", "prompt": "What is the difference between classification and regression? Give one example of each.", "expected": "Classification predicts categories; regression predicts numerical values; examples are appropriate."},
    {"id": 3, "category": "arithmetic", "prompt": "A jacket costs ₹800 and has a 25% discount. What is the final price? Show the calculation.", "expected": "25% of 800 = 200; final price = ₹600."},
    {"id": 4, "category": "arithmetic", "prompt": "A class has 40 students. If 30% are absent, how many students are present? Show the calculation.", "expected": "30% of 40 = 12 absent; 40 − 12 = 28 present."},
    {"id": 5, "category": "ml_concept", "prompt": "What is the purpose of a validation dataset in machine learning?", "expected": "Used during development/model selection or tuning; distinct from training and final test data."},
    {"id": 6, "category": "technical_accuracy", "prompt": "What is data leakage in machine learning, and why is it a problem? Distinguish it from a personal-data privacy leak.", "expected": "Training/evaluation information improperly reveals target or future information, inflating performance; distinguishes this from exposing private data."},
    {"id": 7, "category": "format_following", "prompt": "Explain what a database is in exactly 2 sentences, using simple English.", "expected": "Exactly two sentences; simple and accurate explanation."},
    {"id": 8, "category": "format_following", "prompt": "Give exactly three benefits of unit testing as a numbered list. Do not add an introduction or conclusion.", "expected": "Exactly three numbered benefits; no introductory or concluding text."},
    {"id": 9, "category": "coding", "prompt": "Write a Python function called is_even(n) that returns True if n is even and False otherwise. Return code only.", "expected": "Valid Python function; returns a Boolean based on n % 2 == 0; code only."},
    {"id": 10, "category": "uncertainty", "prompt": "Predict the exact vote share of each party in India's next national election. Explain what you can and cannot know.", "expected": "Does not fabricate exact future results; explains uncertainty and that reliable exact vote shares cannot be known in advance."},
]

SYSTEM_INSTRUCTION = (
    "Answer in clear, simple English. Be accurate and relevant. Follow the user's "
    "requested format and length. If uncertain, state the uncertainty instead of inventing details."
)
with open(OUTPUT_DIR / "test_cases.json", "w", encoding="utf-8") as f:
    json.dump(TEST_CASES, f, ensure_ascii=False, indent=2)
print(f"Loaded {len(TEST_CASES)} test cases")


## 3. Load the model

The notebook uses GPU in half precision when available and CPU otherwise. If Colab cannot download the model or the runtime disconnects, reconnect and rerun from this cell. Do not silently change model or generation settings: record any deliberate change in the final report.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
load_kwargs = {"low_cpu_mem_usage": True}
if torch.cuda.is_available():
    load_kwargs["torch_dtype"] = torch.float16
    load_kwargs["device_map"] = "auto"
else:
    load_kwargs["torch_dtype"] = torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
model_device = next(model.parameters()).device
print("Model loaded:", MODEL_ID)
print("Model device:", model_device)


## 4. Generation function

Latency is measured around generation, excluding model download and initial setup. On CUDA, the device is synchronised before and after generation to make timing more meaningful. Runtime varies by hardware, so compare latency only with the device and runtime noted.


In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_generation(test_case, condition, repeat_index=1, seed=None):
    seed = int(seed if seed is not None else SEED + test_case["id"])
    set_all_seeds(seed)
    messages = []
    if condition == "B_system":
        messages.append({"role": "system", "content": SYSTEM_INSTRUCTION})
    messages.append({"role": "user", "content": test_case["prompt"]})
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(rendered, return_tensors="pt")
    input_device = next(model.parameters()).device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}
    input_tokens = int(inputs["input_ids"].shape[-1])

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    new_token_ids = output[0, input_tokens:]
    answer = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()
    generated_tokens = int(new_token_ids.numel())
    return {
        "test_id": test_case["id"],
        "category": test_case["category"],
        "condition": condition,
        "repeat_index": repeat_index,
        "seed": seed,
        "prompt": test_case["prompt"],
        "expected_answer_notes": test_case["expected"],
        "response": answer,
        "latency_sec": round(elapsed, 4),
        "input_tokens": input_tokens,
        "generated_tokens": generated_tokens,
        "token_limit_hit": generated_tokens >= MAX_NEW_TOKENS,
        "device": str(input_device),
    }


## 5. Run the controlled experiment

The primary A/B run uses the same seed for both conditions for each prompt. With sampling enabled, a shared seed improves control but does not guarantee identical random-number consumption across the different prompt templates. The repeatability subset is run under Condition B with distinct repeat seeds.

This cell performs **26 generations**. Avoid rerunning it unless you intend to replace the current fresh run; rerunning will overwrite the CSV outputs. A copy of the results is written immediately after the run.


In [ ]:
records = []
for case in TEST_CASES:
    paired_seed = SEED + 1000 + case["id"]
    for condition in ["A_baseline", "B_system"]:
        row = run_generation(case, condition, repeat_index=1, seed=paired_seed)
        records.append(row)
        print(f"{condition} | test {case['id']:02d} | {row['latency_sec']:.2f}s | {row['generated_tokens']} tokens")

# Repeat selected Condition B cases twice more to inspect output stability.
for test_id in [1, 6, 7]:
    case = next(c for c in TEST_CASES if c["id"] == test_id)
    for repeat_index in [2, 3]:
        row = run_generation(case, "B_system", repeat_index=repeat_index,
                             seed=SEED + 5000 + test_id * 10 + repeat_index)
        records.append(row)
        print(f"B_system repeat {repeat_index} | test {test_id:02d} | {row['latency_sec']:.2f}s | {row['generated_tokens']} tokens")

results = pd.DataFrame(records)
results.to_csv(OUTPUT_DIR / "raw_generation_results.csv", index=False, encoding="utf-8-sig")
results.to_json(OUTPUT_DIR / "raw_generation_results.jsonl", orient="records", lines=True, force_ascii=False)
print("\nFresh generations:", len(results))
print("Saved raw results to", OUTPUT_DIR.resolve())


## 6. Automatic checks and operational metrics

These checks do not judge factual correctness. They detect empty responses and two narrow format requirements. Sentence splitting and numbered-item detection are heuristic and can make mistakes; inspect responses directly before making claims.


In [ ]:
def count_sentences_heuristic(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip()) if text.strip() else []
    return len([p for p in parts if p.strip()])


def count_numbered_items(text):
    return len(re.findall(r"(?m)^\s*\d+[.)]\s+", text))

results["nonempty"] = results["response"].fillna("").str.strip().ne("")
results["two_sentence_check"] = np.where(
    results["test_id"].eq(7), results["response"].apply(count_sentences_heuristic).eq(2), np.nan
)
results["three_numbered_items_check"] = np.where(
    results["test_id"].eq(8), results["response"].apply(count_numbered_items).eq(3), np.nan
)

primary_for_summary = results[results["repeat_index"] == 1]
summary = primary_for_summary.groupby("condition", dropna=False).agg(
    responses=("response", "size"),
    mean_latency_sec=("latency_sec", "mean"),
    median_latency_sec=("latency_sec", "median"),
    mean_generated_tokens=("generated_tokens", "mean"),
    token_limit_hits=("token_limit_hit", "sum"),
    nonempty_responses=("nonempty", "sum"),
).reset_index()
summary[["mean_latency_sec", "median_latency_sec", "mean_generated_tokens"]] = summary[["mean_latency_sec", "median_latency_sec", "mean_generated_tokens"]].round(3)
summary.to_csv(OUTPUT_DIR / "operational_summary.csv", index=False)
print(summary.to_string(index=False))

format_rows = results[results["test_id"].isin([7, 8])][["test_id", "condition", "repeat_index", "two_sentence_check", "three_numbered_items_check", "response"]]
format_rows.to_csv(OUTPUT_DIR / "format_check_details.csv", index=False)
print("\nFormat check details saved to format_check_details.csv")


## 7. Human audit: quality and instruction following

Automatic checks are not enough to evaluate answer quality. This notebook creates a blinded-ish audit sheet with responses in shuffled order and blank scoring fields. Review the actual responses and score each dimension from **0 to 2**:

- **Factual correctness:** 0 = incorrect/misleading, 1 = partly correct, 2 = correct for the prompt.
- **Relevance and completeness:** 0 = misses the task, 1 = partly answers, 2 = answers the key parts.
- **Instruction following:** 0 = ignores key constraints, 1 = partial compliance, 2 = follows the requested format/length/task.
- **Clarity:** 0 = hard to understand, 1 = somewhat clear, 2 = clear.

Total score = 0–8. Use the expected-answer notes as a guide, not as a substitute for judgment. Keep ratings grounded in the visible response. For a more defensible comparison, score the responses before looking at condition-level summaries, and record any ambiguous decisions in your notes.


In [ ]:
audit = results[["test_id", "category", "condition", "repeat_index", "prompt", "expected_answer_notes", "response"]].copy()
audit["factual_correctness_0_2"] = ""
audit["relevance_completeness_0_2"] = ""
audit["instruction_following_0_2"] = ""
audit["clarity_0_2"] = ""
audit["reviewer_notes"] = ""
audit = audit.sample(frac=1, random_state=SEED).reset_index(drop=True)
audit.to_csv(OUTPUT_DIR / "human_audit_template.csv", index=False, encoding="utf-8-sig")
print("Audit template saved:", OUTPUT_DIR / "human_audit_template.csv")
display(audit[["test_id", "condition", "repeat_index", "prompt", "response"]].head(8))


### Score the audit sheet

Open `human_audit_template.csv` from the output folder, fill the four score columns with integers 0, 1 or 2, and save it back into the same folder with the same filename. In Colab, use the Files panel to download, edit locally, and upload the completed file back to `slm_evaluation_outputs/`. Then run the next cell. If you cannot complete the audit immediately, leave the scores blank and report operational metrics only; do **not** claim that one condition has better quality.


In [ ]:
audit_path = OUTPUT_DIR / "human_audit_template.csv"
audit_scored = pd.read_csv(audit_path)
score_cols = ["factual_correctness_0_2", "relevance_completeness_0_2", "instruction_following_0_2", "clarity_0_2"]
for col in score_cols:
    audit_scored[col] = pd.to_numeric(audit_scored[col], errors="coerce")

if audit_scored[score_cols].notna().all(axis=None):
    valid = audit_scored[score_cols].apply(lambda col: col.between(0, 2)).all(axis=None)
    if not valid:
        raise ValueError("Scores must be integers from 0 to 2.")
    audit_scored["total_quality_score_0_8"] = audit_scored[score_cols].sum(axis=1)
    audit_scored.to_csv(OUTPUT_DIR / "human_audit_scored.csv", index=False, encoding="utf-8-sig")
    human_summary = audit_scored.groupby("condition").agg(
        rated_responses=("total_quality_score_0_8", "count"),
        mean_total_score=("total_quality_score_0_8", "mean"),
        mean_factual_correctness=("factual_correctness_0_2", "mean"),
        mean_relevance_completeness=("relevance_completeness_0_2", "mean"),
        mean_instruction_following=("instruction_following_0_2", "mean"),
        mean_clarity=("clarity_0_2", "mean"),
    ).round(3).reset_index()
    human_summary.to_csv(OUTPUT_DIR / "human_quality_summary.csv", index=False)
    display(human_summary)
else:
    print("Human ratings are incomplete. No human quality comparison has been calculated.")


## 8. Repeatability check

This compares the three Condition B responses for test IDs 1, 6 and 7. Text similarity is a character-sequence similarity score, **not** a semantic similarity or correctness measure. Different wording can still be equally correct, and similar wording can still be wrong.


In [ ]:
repeat_rows = results[(results["condition"] == "B_system") & (results["test_id"].isin([1, 6, 7]))].copy()
stability = []
for test_id, group in repeat_rows.groupby("test_id"):
    group = group.sort_values("repeat_index")
    texts = group["response"].fillna("").tolist()
    similarities = [SequenceMatcher(None, texts[i], texts[j]).ratio()
                    for i in range(len(texts)) for j in range(i + 1, len(texts))]
    stability.append({
        "test_id": test_id,
        "repeats": len(texts),
        "mean_pairwise_text_similarity": round(float(np.mean(similarities)), 3) if similarities else np.nan,
        "latency_range_sec": round(float(group["latency_sec"].max() - group["latency_sec"].min()), 3),
        "token_count_range": int(group["generated_tokens"].max() - group["generated_tokens"].min()),
    })
stability_df = pd.DataFrame(stability)
stability_df.to_csv(OUTPUT_DIR / "repeatability_summary.csv", index=False)
display(stability_df)


## 9. Visualise operational results

Charts below use the freshly recorded outputs. They describe runtime and output length, not answer quality.


In [ ]:
primary = results[results["repeat_index"] == 1].copy()
fig, ax = plt.subplots(figsize=(7, 4))
primary.groupby("condition")["latency_sec"].mean().plot(kind="bar", ax=ax)
ax.set_title("Mean generation latency by condition")
ax.set_ylabel("Seconds (lower is faster)")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mean_latency_by_condition.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
primary.groupby("condition")["generated_tokens"].mean().plot(kind="bar", ax=ax)
ax.set_title("Mean generated tokens by condition")
ax.set_ylabel("Generated tokens")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "mean_tokens_by_condition.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
primary.groupby("condition")["token_limit_hit"].sum().plot(kind="bar", ax=ax)
ax.set_title("Responses reaching the generation token limit")
ax.set_ylabel("Count")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "token_limit_hits_by_condition.png", dpi=150)
plt.show()


## 10. Generate a concise report from observed results

The report below uses only results generated in this run. It will not infer quality superiority from latency or token counts. If human scores have not been completed, the report explicitly says so.


In [ ]:
def safe_mean(series):
    return float(series.mean()) if len(series) else float("nan")

lines = [
    "# SLM Evaluation Report",
    "",
    "## Research question",
    "Does a concise system instruction change the model's behaviour on a fixed set of ten prompts compared with no system message?",
    "",
    "## Setup",
    f"- Model: `{MODEL_ID}`",
    f"- Device: {runtime_info['gpu']}",
    f"- Temperature: {TEMPERATURE}; top-p: {TOP_P}; max new tokens: {MAX_NEW_TOKENS}; sampling: {DO_SAMPLE}",
    f"- Fresh responses: {len(results)} (20 primary A/B responses and 6 repeatability responses expected).",
    "- Latency is generation-only and hardware/runtime dependent.",
    "",
    "## Operational results (primary A/B responses only)",
]
for _, row in summary.iterrows():
    lines.append(f"- {row['condition']}: n={int(row['responses'])}, mean latency={row['mean_latency_sec']:.3f}s, median latency={row['median_latency_sec']:.3f}s, mean generated tokens={row['mean_generated_tokens']:.2f}, token-limit hits={int(row['token_limit_hits'])}.")
lines += ["", "## Human-rated quality", ""]
scored_path = OUTPUT_DIR / "human_audit_scored.csv"
if scored_path.exists():
    scored = pd.read_csv(scored_path)
    needed = ["factual_correctness_0_2", "relevance_completeness_0_2", "instruction_following_0_2", "clarity_0_2"]
    if all(c in scored.columns for c in needed) and scored[needed].notna().all(axis=None):
        scored["total_quality_score_0_8"] = scored[needed].sum(axis=1)
        hs = scored.groupby("condition")["total_quality_score_0_8"].agg(["count", "mean"]).round(3)
        for condition, row in hs.iterrows():
            lines.append(f"- {condition}: {int(row['count'])} rated responses; mean human score {row['mean']}/8.")
    else:
        lines.append("Human audit file exists but is incomplete; no quality superiority claim is made.")
else:
    lines.append("Human scoring has not been completed. No quality superiority claim is made.")
lines += [
    "",
    "## Interpretation and limitations",
    "- This is a small exploratory test set, not a general benchmark.",
    "- The conditions differ by presence of the system instruction; any observed difference is specific to these prompts and settings.",
    "- Automated format checks are heuristic and cover only selected constraints.",
    "- Human ratings are subjective; document ambiguous cases and avoid overstating small score differences.",
    "- Repeated text similarity measures wording overlap, not semantic correctness.",
    "- Latency depends on hardware, model loading, runtime load and library versions.",
    "- Do not compare these results directly with historical experiments that used different scoring rubrics or settings.",
    "",
    "## Conclusion",
    "Complete the human audit and inspect raw responses before concluding whether the system instruction helped. Report the actual observed differences, including cases where it did not help or made an answer worse.",
]
report_text = "\n".join(lines)
(OUTPUT_DIR / "final_report.md").write_text(report_text, encoding="utf-8")
print(report_text)


## 11. Final checks and export

The ZIP contains the raw generations, test definitions, runtime metadata, summaries, charts, audit template and report. Review the raw outputs and complete the human audit before using any quality claim on a CV or in an interview.


In [ ]:
# Refresh raw results and metadata in the export folder.
results.to_csv(OUTPUT_DIR / "raw_generation_results.csv", index=False, encoding="utf-8-sig")
results.to_json(OUTPUT_DIR / "raw_generation_results.jsonl", orient="records", lines=True, force_ascii=False)

zip_path = Path("slm_evaluation_project_outputs.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path))
print("Export created:", zip_path.resolve())

# In Google Colab, uncomment the next two lines to trigger a browser download.
# from google.colab import files
# files.download(str(zip_path))
